# v08 results scorecard

A vanilla reader for the **v08** experiments (assay-filtered + species-stratified), printing the full metric set side-by-side with their **v07** predecessors so the effect of the cleanup is visible.

Metrics:
- **R²** = `(r2-means)²` and **MMD** — from each model's `evals.csv` (ncells=80, nfeatures=all).
- **MMD floor / ceiling / gap_above_floor / frac_gap_closed / mean_JS** — from the `extended_metrics.csv` sidecar written by **`./hub metrics <run_id>`** (run that after the evals land; otherwise those columns show NaN).

This notebook only *reads* CSVs — no model loading, runs in any env with pandas. Add rows to `MODELS` to score more experiments (e.g. the atlas CD8 run).

> The v08 jobs are training as of 2026-06-05; rows will populate as each eval completes. For the extended-metric columns, run e.g. `./hub metrics hvg_pearson_residuals_m1_v08_ood/impact_cellot` once the data-space eval is done.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

REPO = Path("/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT")
RESULTS = REPO / "cellot/cellot_gpu/results"
EVALPREFIX = "evals_ood_data_space"
NCELLS = 80

# (label, results_subdir). Add atlas CD8 etc. here once trained.
MODELS = [
    ("m1 v07 IMPACT", "hvg_pearson_residuals_m1_ood/impact_cellot"),
    ("m1 v07 scGen",  "hvg_pearson_residuals_m1_ood/scgen"),
    ("m1 v08 IMPACT", "hvg_pearson_residuals_m1_v08_ood/impact_cellot"),
    ("m1 v08 scGen",  "hvg_pearson_residuals_m1_v08_ood/scgen"),
    ("m2 v07 IMPACT", "hvg_pearson_residuals_m2_ood/impact_cellot"),
    ("m2 v07 scGen",  "hvg_pearson_residuals_m2_ood/scgen"),
    ("m2 v08 IMPACT", "hvg_pearson_residuals_m2_v08_ood/impact_cellot"),
    ("m2 v08 scGen",  "hvg_pearson_residuals_m2_v08_ood/scgen"),
    # Uncapped FULL-atlas, CD8 holdout: v07 (no assay filter) vs v08 (assay-filtered)
    ("atlas-CD8 v07 IMPACT", "hvg_pearson_residuals_a_ood_uncapped/impact_cellot"),
    ("atlas-CD8 v07 scGen",  "hvg_pearson_residuals_a_ood_uncapped/scgen"),
    ("atlas-CD8 v08 IMPACT", "hvg_pearson_residuals_a_uncapped_v08_ood/impact_cellot"),
    ("atlas-CD8 v08 scGen",  "hvg_pearson_residuals_a_uncapped_v08_ood/scgen"),
]


def _r2_mmd(eval_dir):
    """R2 (= squared r2-means) and MMD at ncells=80 from evals.csv; (nan, nan, status)."""
    p = eval_dir / "evals.csv"
    if not p.exists():
        return np.nan, np.nan, "pending (no evals.csv)"
    df = pd.read_csv(p)
    sub = df[(df["nfeatures"] == "all") & (df["ncells"] == NCELLS)]
    if sub.empty:
        return np.nan, np.nan, "evals.csv has no ncells=80"
    r2 = (sub.loc[sub["metric"] == "r2-means", "value"].astype(float) ** 2).mean()
    mmd = sub.loc[sub["metric"] == "mmd", "value"].astype(float).mean()
    return r2, mmd, "ok"


_EXT_KEYS = ("mmd_floor", "mmd_ceiling", "gap_above_floor", "frac_gap_closed", "mean_js",
             "r2_self", "r2_identity", "frac_r2_closed")


def _extended(eval_dir):
    """Extended metrics (MMD + R2 floor/ceiling) from extended_metrics.csv (max ncells)."""
    p = eval_dir / "extended_metrics.csv"
    if not p.exists():
        return {k: np.nan for k in _EXT_KEYS}
    df = pd.read_csv(p)
    row = df.loc[df["ncells"].idxmax()]
    return {k: (float(row[k]) if k in df.columns and pd.notna(row[k]) else np.nan)
            for k in _EXT_KEYS}


rows = []
for label, sub in MODELS:
    eval_dir = RESULTS / sub / EVALPREFIX
    r2, mmd, status = _r2_mmd(eval_dir)
    ext = _extended(eval_dir)
    rows.append({"model": label, "R2": r2, "MMD": mmd, **ext, "status": status})

scorecard = pd.DataFrame(rows)
numcols = ["R2", "MMD", "mmd_floor", "mmd_ceiling", "gap_above_floor", "frac_gap_closed", "mean_js",
           "r2_self", "r2_identity", "frac_r2_closed"]
scorecard[numcols] = scorecard[numcols].astype(float).round(4)
scorecard.to_csv(REPO / "speciesOT/baseline/analysis/v08_scorecard.csv", index=False)
scorecard

,model,R2,MMD,mmd_floor,mmd_ceiling,gap_above_floor,frac_gap_closed,mean_js,status
0,m1 v07 IMPACT,0.9407,0.1375,0.0238,0.0902,0.1134,-0.7072,0.4653,ok
1,m1 v07 scGen,0.9094,0.1677,0.0238,0.0902,0.1430,-1.1524,0.4901,ok
2,m1 v08 IMPACT,0.9188,0.1035,0.0236,0.1087,0.0800,0.0608,0.4492,ok
3,m1 v08 scGen,0.9270,0.1737,0.0236,0.1087,0.1497,-0.7588,0.5023,ok
4,m2 v07 IMPACT,0.9301,0.1080,0.0237,0.0867,0.0840,-0.3328,0.4507,ok
5,m2 v07 scGen,0.8916,0.1458,0.0237,0.0867,0.1224,-0.9429,0.4736,ok
6,m2 v08 IMPACT,0.9231,0.1146,0.0233,0.1062,0.0935,-0.1276,0.4515,ok
7,m2 v08 scGen,0.9443,0.1610,0.0233,0.1062,0.1389,-0.6747,0.4850,ok
8,atlas-CD8 v07 IMPACT,0.8479,0.0513,NaN,NaN,NaN,NaN,NaN,ok
9,atlas-CD8 v07 scGen,0.9321,0.1163,NaN,NaN,NaN,NaN,NaN,ok


## v07 → v08 deltas

Paired so the cleanup effect is explicit. Reading the signs: **higher R² is better**; **lower MMD / lower gap_above_floor / lower mean_JS are better**. The hope is dropping Smart-seq2 + balancing the split *raises R²* and *lowers the MMD gap*.

In [2]:
def _row(label):
    m = scorecard[scorecard["model"] == label]
    return m.iloc[0] if len(m) else None


def _d(b, a):
    return round(b - a, 4) if (pd.notna(a) and pd.notna(b)) else np.nan


deltas = []
for line in ["m1", "m2", "atlas-CD8"]:
    for mdl in ["IMPACT", "scGen"]:
        a, b = _row(f"{line} v07 {mdl}"), _row(f"{line} v08 {mdl}")
        if a is None or b is None:
            continue
        deltas.append({
            "cell": f"{line} {mdl}",
            "R2_v07": a["R2"], "R2_v08": b["R2"], "dR2": _d(b["R2"], a["R2"]),
            "MMD_v07": a["MMD"], "MMD_v08": b["MMD"], "dMMD": _d(b["MMD"], a["MMD"]),
            "gap_v07": a["gap_above_floor"], "gap_v08": b["gap_above_floor"],
            "dGap": _d(b["gap_above_floor"], a["gap_above_floor"]),
            "JS_v07": a["mean_js"], "JS_v08": b["mean_js"], "dJS": _d(b["mean_js"], a["mean_js"]),
        })

delta_table = pd.DataFrame(deltas)
delta_table.to_csv(REPO / "speciesOT/baseline/analysis/v08_vs_v07_deltas.csv", index=False)
delta_table

,cell,R2_v07,R2_v08,dR2,MMD_v07,MMD_v08,dMMD,gap_v07,gap_v08,dGap,JS_v07,JS_v08,dJS
0,m1 IMPACT,0.9407,0.9188,-0.0219,0.1375,0.1035,-0.0340,0.1134,0.0800,-0.0334,0.4653,0.4492,-0.0161
1,m1 scGen,0.9094,0.9270,0.0176,0.1677,0.1737,0.0060,0.1430,0.1497,0.0067,0.4901,0.5023,0.0122
2,m2 IMPACT,0.9301,0.9231,-0.0070,0.1080,0.1146,0.0066,0.0840,0.0935,0.0095,0.4507,0.4515,0.0008
3,m2 scGen,0.8916,0.9443,0.0527,0.1458,0.1610,0.0152,0.1224,0.1389,0.0165,0.4736,0.4850,0.0114
4,atlas-CD8 IMPACT,0.8479,NaN,NaN,0.0513,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,atlas-CD8 scGen,0.9321,NaN,NaN,0.1163,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
